# 09. Chatbot RAG Agent 구현

## 이 노트북의 실습 구성

이 노트북에서는 **OpenAI Agents SDK**로 채팅 에이전트를 단계별로 확장합니다. 아래 순서대로 진행합니다.

1. **비동기 방식 호출**: 도구 없이 Agent + Runner로 대화만 구현 (메시지 유지, `await Runner.run`)
2. **Streaming 방식 호출**: 같은 에이전트를 `Runner.run_streamed`로 실행해 토큰 단위 실시간 출력
3. **도구를 추가한 Chatbot Agent**: `@function_tool` 날씨 도구 + `WebSearchTool`을 에이전트에 연결
4. **RAG(검색 증강 생성) 도구 추가**: `10_chat-app-rag`의 지식 베이스 검색을 도구로 붙여, 계정·환불·정책 등 FAQ 기반 답변
5. **웹 UI 구현으로 연결**: 동일 RAG를 Streamlit 웹 앱(`10_chat-app-rag/app.py`)으로 구현한 예제 안내

In [2]:
import openai

from dotenv import load_dotenv
load_dotenv() 

Model = "gpt-5-nano"

### 비동기 방식 호출

In [3]:
from agents import Agent, Runner  

messages = []  # 지금까지의 대화 내용을 저장할 리스트

agent = Agent(
    model=Model, 
    name="여행 에이전트",
    instructions="당신은 훌륭한 여행 에이전트입니다. 사용자와 대화하면서 여행 계획을 도와주세요."
)


while True:
    # 사용자 입력 받기
    user_input = input("\n사용자: ")
    
    # 'exit' 입력 시 종료
    if user_input == "exit":
        print("Bye")
        break
    
    # 현재 사용자의 발화를 대화 기록에 추가
    messages.append({"role": "user", "content": user_input})

    response = await Runner.run(agent, input=messages)
    
    # 모델의 응답을 대화 기록에 추가 (assistant 역할)
    messages.append({"role": "assistant", "content": response.final_output})
    
    # 모델의 최종 응답 출력
    print(f"\n여행 에이전트: {response.final_output}")


사용자:  안녕, 난 길동이야



여행 에이전트: 안녕하세요, 길동님! 반가워요. 여행 계획 도와드리겠습니다.

먼저 몇 가지 여쭤봐도 될까요? 답변 주시면 바로 맞춤 계획을 만들어 드릴게요.

- 여행 목적과 유형: 휴양/관광/음식/문화/모험 등 어떤 스타일이 좋으신가요?
- 일정과 기간: 출발 날짜와 총 여행 기간은 어떻게 되나요?
- 예산: 1인당 예산 표준은 어느 정도로 생각하시나요? 항공+숙소 포함 여부도 알려주세요.
- 선호지/선호 방식: 특정 도시나 나라가 있나요, 아니면 제가 후보지를 제안해 드릴까요? 교통은 주로 비행기 위주인가요, 아니면 기차/차량도 고려하나요?
- 숙소 및 활동 선호: 호텔/에어비앤비/리조트 중 선호가 있나요? 선호하는 활동(맛집 탐방, 유적지 방문, 자연 체험, 해양 스포츠 등)이 있다면요.
- 동반인 수와 특별한 요구사항: 가족 여행인지, 알레르기나 식단 제한, 접근성 필요 여부 등이 있나요?
- 비자/백신 등 보험 관련: 필요하신 도움 여부를 알려주세요.

원하시면 제가 바로 맞춤 일정 초안(짧은 2–3일치 미니 플랜이나 5–7일 코스)이나 여러 후보지 목록을 제시해 드릴게요. 어떤 방식으로 시작하시길 원하나요?



사용자:  내 이름이 뭐였지?



여행 에이전트: 네, 이름은 길동님 맞습니다.

다음으로 어떤 여행 계획을 원하시나요? 예를 들면 2–3일 미니 플랜이나 5–7일 코스 중 어떤 방식으로 시작할지 알려주시면 맞춤 제안 드리겠습니다. 또한 여행 유형, 일정, 예산 등도 함께 알려주시면 좋습니다.



사용자:  exit


Bye


### Streaming 방식 호출

In [4]:
#  모델의 답변이 완성될 때까지 기다리지 않고, 한 토큰씩 실시간으로 출력되는 "스트리밍" 방식
from openai.types.responses import ResponseTextDeltaEvent

messages = []  

agent = Agent(
    model=Model,
    name="여행 에이전트",
    instructions="당신은 훌륭한 여행 에이전트입니다. 사용자와 대화하면서 여행 계획을 도와주세요.",
)

while True:
    # 사용자 입력 받기
    user_input = input("\n사용자: ")
    
    # 종료 조건
    if user_input == "exit":
        print("Bye")
        break

    # 현재 사용자 발화를 messages 리스트에 추가
    messages.append({"role": "user", "content": user_input})

    # 에이전트의 답변 출력 시작
    print("\n여행 에이전트: ", end="", flush=True)

    # run_streamed: 모델이 응답을 실시간으로 보낼 수 있도록 함
    # 결과(result)는 이벤트 스트림(event stream)을 포함하며,
    # 이를 async for 루프를 통해 한 토큰씩 처리합니다.
    result = Runner.run_streamed(agent, input=messages)
    full_response = ""  # 전체 응답을 누적할 변수

    # ResponseTextDeltaEvent: 모델이 생성 중인 텍스트 일부를 전달하는 이벤트 타입
    # event.type == "raw_response_event" 일 때,
    # event.data.delta 에는 모델이 방금 생성한 텍스트 조각이 들어 있음
    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            delta = event.data.delta or ""   # 새로 들어온 텍스트 조각
            print(delta, end="", flush=True) # 화면에 즉시 출력
            full_response += delta           # 전체 답변 문자열로 누적

    # 모델의 전체 응답을 대화 기록에 추가
    messages.append({"role": "assistant", "content": full_response})


사용자:  안녕 난 길동이야



여행 에이전트: 안녕하세요, 길동님! 반갑습니다. 저는 여행 계획을 도와주는 AI 여행 에이전트예요. 길동님의 취향과 예산에 맞춘 맞춤 여행을 함께 만들어드릴게요.

먼저 몇 가지를 여쭤봐도 될까요?
- 가고 싶은 목적지나 지역이 있나요? 국내/해외 중 어떤 쪽이 좋으신가요?
- 여행 기간과 가능한 날짜는 언제인가요? 며칠 정도 계획하고 계신지요?
- 몇 명이 함께 가나요? 혼자이신가요, 가족/연인/친구와 함께하시나요?
- 예산은 대략 어느 정도로 잡고 계신가요? 항공+숙소 포함인지, 혹은 항공만 따로 원하시는지도요.
- 선호하는 숙소 스타일은 어떤 게 좋나요? 호텔/리조트/에어비앤비 등
- 교통 수단은 어떤 걸 선호하시나요? 비행기 위주, 기차/버스도 고려, 차 렌트 가능 여부 등
- 관심사나 피하고 싶은 활동이 있나요? (문화 탐방, 맛집, 자연경관, 쇼핑, 가족형 액티비티 등)
- 특별한 제약이나 필요사항이 있나요? 음식 알레르기, 건강 상태, 비자/보험 필요 여부 등

답변 주시면 예산과 기간에 맞춘 2~3개의 맞춤 일정 초안을 바로 준비해 드리겠습니다. 원하시면 지금 바로 인기 테마 3가지의 예시 일정도 간단히 제안해 드릴게요.


사용자:  내 이름이 뭐였지?



여행 에이전트: 맞습니다, 길동님. 이 대화에서는 길동님으로 부를게요. 원하시면 다른 호칭으로도 불러드릴 수 있어요.

다음으로 여행 계획을 시작해볼까요? 혹시 먼저 알고 싶은 정보가 있다면 알려주세요. 아니면 제가 2~3개의 맞춤 일정 초안을 바로 만들어 드리겠습니다.


사용자:  exit


Bye


### 도구를 추가한 Chatbot Agent

In [5]:
import requests
from agents import Agent, Runner, function_tool, WebSearchTool
from openai.types.responses import ResponseTextDeltaEvent

@function_tool
def get_weather(위도: float, 경도: float) -> str:
    print(위도, 경도)
    print(f"Weather 함수 실행 - 도시: {위도, 경도}")
    latitude = 위도
    longitude = 경도
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m")
    data = response.json()
    return data['current']['temperature_2m']

messages = []

agent = Agent(
    model=Model,
    name="여행 에이전트",
    instructions="당신은 훌륭한 여행 에이전트입니다. 사용자와 대화하면서 여행 계획을 도와주세요.",
    tools=[get_weather, WebSearchTool()]
)

while True:
    user_input = input("\n사용자: ")

    if user_input == "exit":
        print("Bye")
        break

    messages.append({"role": "user", "content": user_input})

    print("\n여행 에이전트: ", end="", flush=True)

    result = Runner.run_streamed(agent, input=messages)
    full_response = ""

    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            delta = event.data.delta or ""
            print(delta, end="", flush=True)
            full_response += delta

    messages.append({"role": "assistant", "content": full_response})


사용자:  지금 서울의 기온을 알려줘



여행 에이전트: 37.5665 126.978
Weather 함수 실행 - 도시: (37.5665, 126.978)
서울의 현재 기온은 약 -0.6°C입니다. (화씨로 환산하면 약 30.9°F)

더 필요한 내용이 있나요? 예를 들어 습도나 바람, 강수 여부 등 현황도 함께 알려드릴 수 있어요.


사용자:  exit


Bye


### RAG(검색 증강 생성) 도구 추가

**RAG**는 지식 베이스(FAQ, 문서)를 임베딩해 두고, 사용자 질문과 유사한 문서를 검색한 뒤 그 내용을 맥락으로 답변하는 방식입니다.
에이전트에 **지식 베이스 검색 도구**를 하나 추가하면, 계정·환불·정책 등 관련 질문에 대해 검색 결과를 바탕으로 답할 수 있습니다.
아래에서는 `10_chat-app-rag` 폴더의 `rag` 모듈을 사용해 동일한 검색 로직을 도구로 붙입니다.

In [8]:
from pathlib import Path
import sys

RAG_DIR = Path.cwd() / "10_chat-app-rag"
sys.path.insert(0, str(RAG_DIR))

from agents import Agent, Runner, function_tool, WebSearchTool
from openai.types.responses import ResponseTextDeltaEvent

try:
    from rag import get_retrieved_context
except ImportError:
    get_retrieved_context = None

@function_tool
def search_knowledge_base(query: str) -> str:
    """사용자 질문에 맞는 FAQ/문서 맥락을 검색합니다.
    계정, 비밀번호, 환불, 고객지원, 보안, 회사 정책 관련 질문일 때 이 도구를 사용하세요."""
    if get_retrieved_context is None:
        return "RAG 모듈을 불러올 수 없습니다. 10_chat-app-rag 폴더가 있고 OPENAI_API_KEY가 설정되어 있는지 확인하세요."
    return get_retrieved_context(query, "openai")


messages = []

tools_list = [get_weather, WebSearchTool()]
if get_retrieved_context is not None:
    tools_list.append(search_knowledge_base)

print(tools_list)

agent = Agent(
    model=Model,
    name="여행·지식 베이스 에이전트",
    instructions=(
        "당신은 여행 계획과 회사 FAQ를 도와주는 에이전트입니다. "
        "날씨가 필요하면 get_weather, 최신 정보는 WebSearchTool, "
        "계정·환불·정책 등은 search_knowledge_base로 지식 베이스를 검색한 뒤 답하세요. "
        "답은 간결하게 하세요."
    ),
    tools=tools_list,
)

while True:
    user_input = input("\n사용자: ")

    if user_input == "exit":
        print("Bye")
        break

    messages.append({"role": "user", "content": user_input})

    print("\n에이전트: ", end="", flush=True)

    result = Runner.run_streamed(agent, input=messages)
    full_response = ""

    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            delta = event.data.delta or ""
            print(delta, end="", flush=True)
            full_response += delta

    messages.append({"role": "assistant", "content": full_response})

[FunctionTool(name='get_weather', description='', params_json_schema={'properties': {'위도': {'title': '위도', 'type': 'number'}, '경도': {'title': '경도', 'type': 'number'}}, 'required': ['위도', '경도'], 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000022761B2C4A0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False), WebSearchTool(user_location=None, filters=None, search_context_size='medium'), FunctionTool(name='search_knowledge_base', description='사용자 질문에 맞는 FAQ/문서 맥락을 검색합니다.\n계정, 비밀번호, 환불, 고객지원, 보안, 회사 정책 관련 질문일 때 이 도구를 사용하세요.', params_json_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_knowledge_base_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<l


사용자:  지금 부산 날씨가 어때?



에이전트: 35.16 129.06
Weather 함수 실행 - 도시: (35.16, 129.06)
부산의 현재 기온은 약 4.7°C입니다. 원하시면 바람, 습도, 강수 확률 등 자세한 현황이나 오늘의 예보도 같이 알려드릴게요.


사용자:  최신 LangChain version 이 얼마야?



에이전트: 다음이 현재 시점의 최신 버전입니다 (언어별로 구분).

- Python LangChain (PyPI: langchain): 1.2.9, 2026-02-06 release. ([pypi.org](https://pypi.org/project/langchain/?utm_source=openai))
- JavaScript/TypeScript LangChain.js (npm: langchain): 0.3.33, 약 2026-02-11 배포 추정. ([npmjs.com](https://www.npmjs.com/package/langchain?utm_source=openai))

참고로 LangChain JS에서도 1.0 마일스톤 발표가 있었고(2025-10-22 블로그), 이후 버전 관리 방향이 바뀌었습니다. 자세한 내용은 관련 블로그와 문서를 확인해 보세요. ([blog.langchain.com](https://blog.langchain.com/langchain-langgraph-1dot0/?utm_source=openai))

원하시면 사용하려는 언어를 알려주셔서 설치 명령과 간단한 예제까지 바로 알려드리겠습니다.


사용자:  우리회사 환불 정책에 대해 말해줘



에이전트: 다음이 우리 회사의 환불 정책 요약입니다:

- 구매일로부터 30일 이내 환불 가능. 요청 시 주문 번호를 함께 고객 지원팀에 문의하세요.
- 전화 상담 운영 시간: 월요일~금요일 10:00–18:00 (IST).

필요하시면 현재 정책의 예외 규정이나 환불 절차를 더 자세히 정리해 드리겠습니다. 특정 주문 정보를 가지고 계시면 알려주시면 바로 안내해 드릴게요.


사용자:  exit


Bye


---------------
### 웹 UI 구현으로 연결

위에서 사용한 **RAG 도구**와 동일한 방식을 **Streamlit 웹 앱**으로 구현한 예제가 **`01_OpenAI_API_Basic/10_chat-app-rag/app.py`** 입니다.
해당 앱에서는 채팅 화면, OpenAI로 답변 생성, 대화 기록 유지까지 포함하므로,
노트북에서 익힌 RAG 도구가 웹에서 어떻게 쓰이는지 확인할 수 있습니다.

실행: `10_chat-app-rag` 폴더에서 `streamlit run app.py`